In [ ]:
import torch
import os
import pretty_midi
import numpy as np
import sys
import argparse

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from m2a_transformer import RoFormerSymbolicTransformer, SOS_TOKEN, EOS_TOKEN, PAD_TOKEN
from preprocess_large_midi_dataset import preprocess_midi, DURATION_TEMPLATES


In [ ]:
model_path = '../ckpt/cp_transformer_909+ac+1k7_trackemb_interleavepos_v0.2_large_batch_40_schedule.epoch=00.val_loss=0.90296.ckpt'
print(f"Loading model from: {model_path}")
# Determine model size from checkpoint path name
if 'small' in model_path:
    model = RoFormerSymbolicTransformer.load_from_checkpoint(model_path, large=False)
else:
    model = RoFormerSymbolicTransformer.load_from_checkpoint(model_path, large=True)

In [ ]:
model.cuda()
model.eval()
print("Model loaded successfully.")

In [ ]:
def notes_to_rolls(notes, max_tick, max_polyphony=4, program=0):
    """
    Converts a list of note events to a tensor in the format expected by the model.

    Args:
        notes (list[dict]): A list of note events. Each note is a dictionary 
                            with 'tick', 'pitch', and 'duration'.
        max_tick (int): The total number of ticks for the resulting tensor.
        max_polyphony (int): The maximum number of simultaneous notes allowed.
        program (int): The program number to assign to the notes.

    Returns:
        torch.Tensor: A tensor representing the notes in the model's input format.
    """
    duration_boundaries = (DURATION_TEMPLATES[1:] + DURATION_TEMPLATES[:-1]) / 2.0
    rolls = np.full((max_tick, max_polyphony, 3), dtype=np.uint8, fill_value=255)
    polyphony_counts = np.zeros(max_tick, dtype=np.uint8)

    for note in notes:
        tick = note['tick']
        if tick >= max_tick:
            continue

        if polyphony_counts[tick] >= max_polyphony:
            print(f"Warning: Exceeded max polyphony at tick {tick}. Note with pitch {note['pitch']} dropped.")
            continue
        
        duration_idx = np.searchsorted(duration_boundaries, note['duration'])

        slot = polyphony_counts[tick]
        rolls[tick, slot, 0] = program
        rolls[tick, slot, 1] = note['pitch']
        rolls[tick, slot, 2] = duration_idx
        
        polyphony_counts[tick] += 1

    for i in range(max_tick):
        if polyphony_counts[i] > 1:
            sorted_indices = np.argsort(rolls[i, :polyphony_counts[i], 1])
            rolls[i, :polyphony_counts[i]] = rolls[i, :polyphony_counts[i]][sorted_indices]

    return torch.tensor(rolls.reshape(max_tick, -1))

def tensors_to_notes(output_tensors, single=False):
    """
    Decodes the symbolic output from the model into lists of note events.

    Args:
        output_tensors (list[torch.Tensor]): A list of tensors from the model's output.
        single (bool): Whether the output is interleaved.

    Returns:
        list[list[dict]]: A list of note lists, one for each generated sample.
    """
    notes_per_sample = []
    num_timesteps = len(output_tensors)
    if num_timesteps == 0:
        return []
    
    n_samples = output_tensors[0].shape[0]

    for i in range(n_samples):
        sample_notes = []
        for time_step in range(num_timesteps):
            data = output_tensors[time_step][i]
            content = data.flatten()
            tick = time_step if single else time_step // 2
            
            for j in range(0, len(content), 2):
                program = int(content[j].item())
                if program == EOS_TOKEN or program == PAD_TOKEN:
                    continue
                if j + 1 >= len(content):
                    break
                
                pitch_duration = int(content[j+1].item()) - 2
                if pitch_duration < 0:
                    continue

                pitch = pitch_duration % 128
                duration_idx = pitch_duration // 128

                if not (0 <= duration_idx < len(DURATION_TEMPLATES)):
                    continue

                duration = DURATION_TEMPLATES[duration_idx]

                sample_notes.append({
                    'tick': tick,
                    'pitch': pitch,
                    'duration': int(duration),
                    'program': program
                })
        notes_per_sample.append(sample_notes)
    return notes_per_sample

def decode_output_to_notes(outputs, single=False):
    """
    Decodes the symbolic output from the model into a list of note events.

    Args:
        outputs (list[torch.Tensor] or tuple[torch.Tensor]): A list or tuple of tensors, where each tensor 
                                                              represents a generated musical frame.
        single (bool, optional): If True, each time step in the output corresponds to one time step.
                                 If False, it's assumed the output is interleaved (e.g., acc, mel, acc, mel),
                                 so the time step is halved. Defaults to False.
    Returns:
        list[dict]: A list of note event dictionaries. Each dictionary contains 'tick', 'program', 'pitch', and 'duration'.
    """
    notes = []
    
    if not isinstance(outputs, tuple):
        outputs = (outputs,)

    for output in outputs:
        for time_step, data in enumerate(output):
            content = data.squeeze(0)
            tick = time_step if single else time_step // 2

            for i in range(0, len(content), 2):
                program = int(content[i].item())
                
                if program == EOS_TOKEN:
                    break
                if i + 1 >= len(content):
                    break
                
                pitch_duration = int(content[i+1].item()) - 2
                if pitch_duration < 0: continue

                pitch = pitch_duration % 128
                duration_idx = pitch_duration // 128

                if program not in [0, 1]:
                    continue
                if not (0 <= pitch < 128):
                    continue
                if not (0 <= duration_idx < len(DURATION_TEMPLATES)):
                    continue
                
                duration_in_ticks = DURATION_TEMPLATES[duration_idx]

                notes.append({
                    'tick': tick,
                    'program': program,
                    'pitch': pitch,
                    'duration': int(duration_in_ticks)
                })
    return notes

def generate_from_notes(model, melody_notes, accompaniment_notes=[], prompt_length_ticks=100, generation_length_frames=384, temperature=1.0, n_samples=1, max_polyphony=4):
    """
    Generates a musical continuation from lists of note events.

    Args:
        model: The trained music generation model.
        melody_notes (list[dict]): A list of melody note events.
        accompaniment_notes (list[dict]): An optional list of accompaniment note events.
        prompt_length_ticks (int): The number of ticks from the input notes to use as a prompt.
        generation_length_frames (int): The number of frames (time steps) to generate.
        temperature (float): The sampling temperature.
        n_samples (int): The number of samples to generate.
        max_polyphony (int): The maximum polyphony.

    Returns:
        list[list[dict]]: A list of generated note sequences.
    """
    all_notes = melody_notes + accompaniment_notes
    if not all_notes:
        print("No notes provided.")
        return []

    max_tick = max(n['tick'] for n in all_notes) if all_notes else 0
    prompt_ticks = min(prompt_length_ticks, max_tick + 1)

    x_mel_raw = notes_to_rolls(melody_notes, prompt_ticks, max_polyphony, program=0)
    x_acc_raw = notes_to_rolls(accompaniment_notes, prompt_ticks, max_polyphony, program=1)

    x_mel_raw = x_mel_raw.unsqueeze(0).cuda()
    x_acc_raw = x_acc_raw.unsqueeze(0).cuda()

    x_mel, x_acc = model.preprocess(x_mel_raw.long(), pitch_shift=torch.zeros(1, dtype=torch.int8).cuda(), y=x_acc_raw.long())

    batch_size, seq_len, subseq_len = x_mel.shape
    stacked = torch.stack([x_acc, x_mel], dim=2)
    x = stacked.view(batch_size, seq_len * 2, subseq_len)

    with torch.no_grad():
        x = x.repeat(n_samples, 1, 1)
        output_tensors = model.global_sampling(x, x_mel_gt=None, temperature=temperature, max_seq_len=generation_length_frames)

    all_samples_notes = tensors_to_notes(output_tensors)

    return all_samples_notes

In [ ]:
# Example usage:
melody_notes = [
    {'tick': 0, 'pitch': 60, 'duration': 4},
    {'tick': 4, 'pitch': 62, 'duration': 4},
    {'tick': 8, 'pitch': 64, 'duration': 4},
]

accompaniment_notes = [
    {'tick': 0, 'pitch': 48, 'duration': 8},
    {'tick': 8, 'pitch': 50, 'duration': 8},
]

generated_notes = generate_from_notes(
    model,
    melody_notes,
    accompaniment_notes,
    prompt_length_ticks=16,
    generation_length_frames=20
)

# Print the first generated sample
print(generated_notes[0])